In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../../tmp_data/merged_part_1.csv")

df = df[
    [
        "id",
        "ceiling_height",
        "metro_minutes",
        "metro_walking",
        "total_area",
        "living_area",
        "kitchen_area",
        "price",
        "utilities_amount",
        "utilities_included",
        "prepayment_months",
        "is_long_rental_term",
    ]
]

df.head()

,id,ceiling_height,metro_minutes,metro_walking,total_area,living_area,kitchen_area,price,utilities_amount,utilities_included,prepayment_months,is_long_rental_term
0,271271157,3.0,9,1,200.0,20.0,NaN,500000.0,0.0,1,1.0,1
1,271634126,3.5,8,1,198.0,95.0,18.0,500000.0,0.0,1,1.0,1
2,271173086,3.2,7,1,200.0,116.0,4.0,500000.0,0.0,1,1.0,1
3,272197456,3.2,3,1,170.0,95.0,17.0,400000.0,0.0,1,1.0,1
4,273614615,3.9,7,1,58.0,38.0,5.0,225000.0,0.0,1,1.0,1


In [2]:
# Высота потолков

# Процент пропусков
print("Процент пропусков в ceiling_height: ", df["ceiling_height"].isna().mean() * 100)

# Флаг до всего
df["ceiling_height_known"] = df["ceiling_height"].notna().astype(int)

valid_heights = df["ceiling_height"].dropna()
LOWER = valid_heights.quantile(0.01)
UPPER = valid_heights.quantile(0.99)
MEDIAN = valid_heights.median()

outliers_mask = (df["ceiling_height"] < LOWER) | (df["ceiling_height"] > UPPER)
cleaned = df["ceiling_height"].copy()
cleaned[outliers_mask] = np.nan
df["ceiling_height"] = cleaned.fillna(MEDIAN)

print(f"Высота потолков: от {LOWER:.2f} до {UPPER:.2f} м, медиана {MEDIAN:.2f} м")
print(
    f"Выбросов: {outliers_mask.sum()}, пропусков: {df['ceiling_height'].isna().sum()}"
)

Процент пропусков в ceiling_height:  51.63829316825116
Высота потолков: от 2.48 до 3.80 м, медиана 2.64 м
Выбросов: 146, пропусков: 0


In [3]:
# Метро

# Считаем количество нулей (значения, где не было цифр)
zeros_count = (df["metro_minutes"] == 0).sum()
print(f"Всего нулей (где время не указано): {zeros_count}")

# Вычисляем среднее время отдельно для пешком и для машины
# Округляем до целых чисел, чтобы сохранить тип int
median_walking = int(round(df[df["metro_walking"] == 1]["metro_minutes"].median()))
median_car = int(round(df[df["metro_walking"] == 0]["metro_minutes"].median()))

print(f"Медианное время пешком (округлено): {median_walking} мин")
print(f"Медианное время на машине (округлено): {median_car} мин")

# Заменяем нули на средние значения (с учетом типа передвижения)
df.loc[(df["metro_minutes"] == 0) & (df["metro_walking"] == 1), "metro_minutes"] = (
    median_walking
)
df.loc[(df["metro_minutes"] == 0) & (df["metro_walking"] == 0), "metro_minutes"] = (
    median_car
)

# Проверяем, что все нули заменены
print(f"\nОсталось нулей после замены: {(df['metro_minutes'] == 0).sum()}")

# Смотрим итоговую статистику
print("\nНовая статистика:")
print(df[["metro_minutes", "metro_walking"]].describe())

Всего нулей (где время не указано): 2317
Медианное время пешком (округлено): 8 мин
Медианное время на машине (округлено): 7 мин

Осталось нулей после замены: 0

Новая статистика:
       metro_minutes  metro_walking
count   22615.000000   22615.000000
mean       11.841433       0.885032
std        63.275716       0.318990
min         1.000000       0.000000
25%         6.000000       1.000000
50%         8.000000       1.000000
75%        13.000000       1.000000
max      1905.000000       1.000000


In [4]:
print("Значения", df["prepayment_months"].value_counts(dropna=False))
print(
    "Процент пропусков в prepayment_months: ",
    df["prepayment_months"].isna().mean() * 100,
)

df["prepayment_months"] = df["prepayment_months"].fillna(
    df["prepayment_months"].mode()[0]
)

Значения prepayment_months
1.0     21790
NaN       541
2.0       267
3.0        10
12.0        5
11.0        1
6.0         1
Name: count, dtype: int64
Процент пропусков в prepayment_months:  2.392217554720318


In [5]:
df.isna().sum()

id                         0
ceiling_height             0
metro_minutes              0
metro_walking              0
total_area                 0
living_area             2876
kitchen_area            5139
price                      0
utilities_amount           0
utilities_included         0
prepayment_months          0
is_long_rental_term        0
ceiling_height_known       0
dtype: int64

In [6]:
df.to_csv("../../tmp_data/bek_processed_part_2.csv", index=False)
df.head()

,id,ceiling_height,metro_minutes,metro_walking,total_area,living_area,kitchen_area,price,utilities_amount,utilities_included,prepayment_months,is_long_rental_term,ceiling_height_known
0,271271157,3.00,9,1,200.0,20.0,NaN,500000.0,0.0,1,1.0,1,1
1,271634126,3.50,8,1,198.0,95.0,18.0,500000.0,0.0,1,1.0,1,1
2,271173086,3.20,7,1,200.0,116.0,4.0,500000.0,0.0,1,1.0,1,1
3,272197456,3.20,3,1,170.0,95.0,17.0,400000.0,0.0,1,1.0,1,1
4,273614615,2.64,7,1,58.0,38.0,5.0,225000.0,0.0,1,1.0,1,1
